# Data Cleaning and Preparation 

### Objectives: 

Prepare the dataset for growth, engagement, funnel and revenue analysis. 

Key tasks: 
- Assess data quality 
- Check duplicates 
- Analyse missing values
- Convert timestamps 
- Create time-based features
- Validate numerical fields

In [2]:
import pandas as pd 
import numpy as np

### Load Sample Dataset

In [3]:
df = pd.read_csv(
    r'D:/Product_Analytics/product_growth_analytics/data/Oct_2019.csv',
    nrows = 1000000)

### Dataset Shape

In [7]:
df.shape

(1000000, 9)

### Duplicate Analysis

In [8]:
df.duplicated().sum()

np.int64(502)


### Duplicate Removal

In [9]:
df.shape

(1000000, 9)

In [10]:
df = df.drop_duplicates()

df.shape

(999498, 9)

In [11]:
df.duplicated().sum()

np.int64(0)

### Observation 

Duplicated records were identified and removed. 

### Business Impact

Removing duplicate events improve the accuracy of the subsequent analyses as they can artificially inflat the user activity, conversion and revenue metrics.

### Missing Value Analysis

In [12]:
df.isnull().sum()

event_time            0
event_type            0
product_id            0
category_id           0
category_code    318053
brand            147550
price                 0
user_id               0
user_session          0
dtype: int64

### Observations 

brand and category_code contain missing values 

### Business impact 

These fields are mainly used for product categorization and segmentation. Missing values in these fields do not prevent user-level and funnel-level analysis.

### Data Validation

In [13]:
df.info()

<class 'pandas.DataFrame'>
Index: 999498 entries, 0 to 999999
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   event_time     999498 non-null  str    
 1   event_type     999498 non-null  str    
 2   product_id     999498 non-null  int64  
 3   category_id    999498 non-null  int64  
 4   category_code  681445 non-null  str    
 5   brand          851948 non-null  str    
 6   price          999498 non-null  float64
 7   user_id        999498 non-null  int64  
 8   user_session   999498 non-null  str    
dtypes: float64(1), int64(3), str(5)
memory usage: 76.3 MB


### Observation 

event_time is currently stored as an object (str) and requires conversion to datetime format before time-based analysis.

### Timestamp Conversion

In [14]:
df["event_time"] = pd.to_datetime(df["event_time"]) 

df.info()

<class 'pandas.DataFrame'>
Index: 999498 entries, 0 to 999999
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype              
---  ------         --------------   -----              
 0   event_time     999498 non-null  datetime64[us, UTC]
 1   event_type     999498 non-null  str                
 2   product_id     999498 non-null  int64              
 3   category_id    999498 non-null  int64              
 4   category_code  681445 non-null  str                
 5   brand          851948 non-null  str                
 6   price          999498 non-null  float64            
 7   user_id        999498 non-null  int64              
 8   user_session   999498 non-null  str                
dtypes: datetime64[us, UTC](1), float64(1), int64(3), str(4)
memory usage: 76.3 MB


### Observation 

event_time successfully converted to datetime format.

In [15]:
df["date"] = df["event_time"].dt.date 

df["hour"] = df["event_time"].dt.hour

df["day"] = df["event_time"].dt.day_name()

df["month"] = df["event_time"].dt.month 

df.info()

<class 'pandas.DataFrame'>
Index: 999498 entries, 0 to 999999
Data columns (total 13 columns):
 #   Column         Non-Null Count   Dtype              
---  ------         --------------   -----              
 0   event_time     999498 non-null  datetime64[us, UTC]
 1   event_type     999498 non-null  str                
 2   product_id     999498 non-null  int64              
 3   category_id    999498 non-null  int64              
 4   category_code  681445 non-null  str                
 5   brand          851948 non-null  str                
 6   price          999498 non-null  float64            
 7   user_id        999498 non-null  int64              
 8   user_session   999498 non-null  str                
 9   date           999498 non-null  object             
 10  hour           999498 non-null  int32              
 11  day            999498 non-null  str                
 12  month          999498 non-null  int32              
dtypes: datetime64[us, UTC](1), float64(1), int32(

### Purpose of new columns 

- **date** : used for DAU calculation
- **hour** : used to identify peak traffic hours
- **day_name** : used to identify active weekdays
- **month** : used for monthly trend analysis

### Price Validation 

In [16]:
(df["price"] < 0).sum()

np.int64(0)

In [17]:
df["price"].describe()

count    999498.000000
mean        295.984458
std         368.244819
min           0.000000
25%          64.320000
50%         161.930000
75%         360.370000
max        2574.070000
Name: price, dtype: float64

### Observation

Price values are reviewed for anomalies and negative values. There are no negative values. 

min price = 0. This needs investigation as we are not sure if it is a promotional product or a missing value encoded as 0 or a data quality issue.

The price distribution seems positively-skewed as the mean price is higher than the P50 (median price). 

This suggests that while most products are relatively lower priced, a small number of high-priced products increase the average price. The presence of substantial variability in product prices are also supported by high standard deviation.

### Event Distribution Validation

In [18]:
df["event_type"].value_counts()

event_type
view        968453
purchase     16846
cart         14199
Name: count, dtype: int64

### Observation 

The majority of events are product views whereas cart additions and purchases occur much less frequently. 

### Business Insight 

This indicates a natural conversion funnel with potential-dropoffs between browsing, cart addition and purchase stages. 

## Data Cleaning Summary 

### Actions performed 

- Checked for duplicate records 
- Assessed missing values
- Validated datatypes
- Converted timestamps to datetime format 
- Created time-based features
- Validated price information
- Reviewed event distribution

### Key Observations 

- Duplicate records identified and removed
- Missing values identified in volumns category_code and brand
- No negative values found in prices
- Price distribution seems right-skewed